# Phase 6: Model Fine-Tuning (Unfrozen)

In this phase, we take the model that was trained in Phase 5 with the frozen convolutional base and **unfreeze** all of its layers. This allows the model to fine-tune its feature extractors to specifically look for details relevant to carabiners (like gate variations or locking mechanisms), rather than just using general features from ImageNet.

We will:
1. **Load** the best weights from Phase 5.
2. **Unfreeze** all model layers (`requires_grad = True`).
3. **Train** for 15 additional epochs with a lower learning rate (`1e-4`) to prevent catastrophic forgetting.
4. Use a **Learning Rate Scheduler** (`CosineAnnealingLR`) to gradually decay the learning rate, helping the model converge to a better minimum.

### 1. Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import time
import copy
from pathlib import Path
import os

# Set device 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

# Paths
PROCESSED_DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = MODELS_DIR / 'best_frozen_resnet34.pth'

mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]

# Redefine Transforms (same as Phase 5)
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
}

# Dataloaders
image_datasets = {x: datasets.ImageFolder(root=PROCESSED_DATA_DIR / x, transform=data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=(x == 'train'), num_workers=0) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Classes: {class_names}")
print(f"Dataset Sizes: {dataset_sizes}")

### 2. Model Configuration & Loading Saved Weights

In [ ]:
from torchvision.models import ResNet34_Weights

# 1. Initialize the structure of ResNet34
model = models.resnet34(weights=None) # We don't need purely ImageNet weights anymore since we are loading our checkpoint
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

# 2. Load the best frozen weights from Phase 5
print(f"Loading Phase 5 checkpoint from: {CHECKPOINT_PATH}")
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

# 3. UNFREEZE all layers for fine-tuning
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

# Criteria
criterion = nn.CrossEntropyLoss()

# Optimizer: We now pass ALL parameters (`model.parameters()`) to the optimizer
# NOTE: We dramatically reduce the learning rate to 1e-4 to avoid wrecking the pre-trained weights
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Learning Rate Scheduler: CosineAnnealingLR smoothly decreases the learning rate
num_epochs = 15
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

### 3. Training Loop (Fine-Tuning)

In [ ]:
def train_model_finetune(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
            else:
                model.eval()   

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
            # Step the learning rate scheduler after the training phase of each epoch
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

### 4. Execute Fine-Tuning

In [ ]:
model_finetuned, training_history = train_model_finetune(model, criterion, optimizer, scheduler, num_epochs=num_epochs)

# Save the final fine-tuned weights
save_path = MODELS_DIR / 'best_finetuned_resnet34.pth'
torch.save(model_finetuned.state_dict(), save_path)
print(f"\nSaved best fine-tuned model weights to: {save_path}")